[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_04/NB_bishop_ch04_linear_regression.ipynb)

# Chapter 4: Single-layer Networks -- Regression

This notebook implements linear regression with polynomial basis functions from Bishop's *Deep Learning: Foundations and Concepts*, Chapter 4: MLE solution, regularized least squares (ridge regression), and a bias-variance decomposition experiment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)

## 1. Generate Synthetic Regression Data

In [ ]:
np.random.seed(42)

def true_function(x):
    return np.sin(2 * np.pi * x)

N = 25
sigma = 0.3
x_train = np.sort(np.random.uniform(0, 1, N))
t_train = true_function(x_train) + np.random.normal(0, sigma, N)

x_test = np.linspace(0, 1, 200)
t_true = true_function(x_test)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_train, t_train, facecolors='none', edgecolors='#1a3a6e',
           s=60, zorder=5, label='Training data')
ax.plot(x_test, t_true, '--', color='#228B22', linewidth=1.5,
        label=r'$\sin(2\pi x)$')
ax.set_xlabel('x')
ax.set_ylabel('t')
ax.set_title('Synthetic Regression Data')
ax.legend()
fig.tight_layout()
plt.show()

## 2. Basis Functions

We consider three types of basis functions:
- **Polynomial**: $\phi_j(x) = x^j$
- **Gaussian RBF**: $\phi_j(x) = \exp\left(-\frac{(x - \mu_j)^2}{2s^2}\right)$
- **Sigmoidal**: $\phi_j(x) = \sigma\left(\frac{x - \mu_j}{s}\right)$

In [ ]:
def polynomial_basis(x, M):
    """Polynomial basis functions."""
    return np.vander(x, M + 1, increasing=True)

def gaussian_basis(x, M, s=0.1):
    """Gaussian RBF basis functions."""
    centers = np.linspace(0, 1, M)
    Phi = np.ones((len(x), M + 1))  # include bias
    for j, mu_j in enumerate(centers):
        Phi[:, j + 1] = np.exp(-0.5 * ((x - mu_j) / s) ** 2)
    return Phi

def sigmoid_basis(x, M, s=0.1):
    """Sigmoidal basis functions."""
    centers = np.linspace(0, 1, M)
    Phi = np.ones((len(x), M + 1))  # include bias
    for j, mu_j in enumerate(centers):
        Phi[:, j + 1] = 1.0 / (1.0 + np.exp(-(x - mu_j) / s))
    return Phi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x_dense = np.linspace(0, 1, 300)
M = 6

bases = [
    ('Polynomial', polynomial_basis(x_dense, M)),
    ('Gaussian RBF', gaussian_basis(x_dense, M, s=0.1)),
    ('Sigmoidal', sigmoid_basis(x_dense, M, s=0.05))
]

for ax, (name, Phi) in zip(axes, bases):
    for j in range(Phi.shape[1]):
        ax.plot(x_dense, Phi[:, j], linewidth=1.5)
    ax.set_xlabel('x')
    ax.set_ylabel(r'$\phi_j(x)$')
    ax.set_title(f'{name} Basis (M={M})')

fig.suptitle('Bishop Ch 4: Basis Functions for Linear Regression', fontsize=13, y=1.03)
fig.tight_layout()
save_fig(fig, 'bishop_ch4_basis_functions')
plt.show()

## 3. Maximum Likelihood Solution

$$\mathbf{w}_{\text{ML}} = (\Phi^T \Phi)^{-1} \Phi^T \mathbf{t}$$

In [ ]:
def fit_ml(Phi, t):
    """Maximum likelihood (ordinary least squares) solution."""
    return np.linalg.lstsq(Phi, t, rcond=None)[0]

def fit_regularized(Phi, t, lam):
    """Regularized least squares (ridge regression)."""
    I = np.eye(Phi.shape[1])
    return np.linalg.solve(Phi.T @ Phi + lam * I, Phi.T @ t)

In [ ]:
# Compare basis functions for regression
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
M_fit = 9

basis_configs = [
    ('Polynomial', polynomial_basis(x_train, M_fit), polynomial_basis(x_test, M_fit)),
    ('Gaussian RBF', gaussian_basis(x_train, M_fit, 0.1), gaussian_basis(x_test, M_fit, 0.1)),
    ('Sigmoidal', sigmoid_basis(x_train, M_fit, 0.05), sigmoid_basis(x_test, M_fit, 0.05))
]

for ax, (name, Phi_tr, Phi_te) in zip(axes, basis_configs):
    w = fit_ml(Phi_tr, t_train)
    y_pred = Phi_te @ w

    ax.scatter(x_train, t_train, facecolors='none', edgecolors='#1a3a6e',
               s=40, zorder=5)
    ax.plot(x_test, t_true, '--', color='#228B22', linewidth=1.5)
    ax.plot(x_test, y_pred, color='#cd0000', linewidth=2)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.8, 1.8)
    ax.set_xlabel('x')
    ax.set_ylabel('t')
    ax.set_title(f'{name} (M={M_fit})')

fig.suptitle('Bishop Ch 4: ML Fit with Different Bases', fontsize=13, y=1.03)
fig.tight_layout()
plt.show()

## 4. Regularized Least Squares (Ridge Regression)

$$\mathbf{w}^* = (\Phi^T \Phi + \lambda I)^{-1} \Phi^T \mathbf{t}$$

In [ ]:
M_reg = 9
Phi_train = gaussian_basis(x_train, M_reg, s=0.1)
Phi_test = gaussian_basis(x_test, M_reg, s=0.1)

log_lambdas = [-30, -6, -3, 0]
labels_reg = [r'$\ln\lambda = -30$ (no reg.)', r'$\ln\lambda = -6$',
              r'$\ln\lambda = -3$', r'$\ln\lambda = 0$']
colors_reg = ['#cd0000', '#FF8C00', '#228B22', '#1a3a6e']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, ln_lam, label, c in zip(axes.ravel(), log_lambdas, labels_reg, colors_reg):
    lam = np.exp(ln_lam)
    w = fit_regularized(Phi_train, t_train, lam)
    y_pred = Phi_test @ w

    ax.scatter(x_train, t_train, facecolors='none', edgecolors='#1a3a6e',
               s=40, zorder=5, label='Data')
    ax.plot(x_test, t_true, '--', color='#228B22', linewidth=1.5,
            label=r'$\sin(2\pi x)$')
    ax.plot(x_test, y_pred, color=c, linewidth=2, label=label)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.8, 1.8)
    ax.set_xlabel('x')
    ax.set_ylabel('t')
    ax.set_title(label)
    ax.legend(fontsize=7)

fig.suptitle('Bishop Ch 4: Regularized Regression (Gaussian RBF Basis)', fontsize=13, y=1.02)
fig.tight_layout()
save_fig(fig, 'bishop_ch4_regularized')
plt.show()

## 5. Regularization Path: RMSE vs lambda

In [ ]:
def rmse(y_pred, y_true):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

# Generate separate test data
np.random.seed(99)
x_val = np.sort(np.random.uniform(0, 1, 100))
t_val = true_function(x_val) + np.random.normal(0, sigma, 100)
Phi_val = gaussian_basis(x_val, M_reg, s=0.1)

ln_lam_range = np.linspace(-35, 5, 100)
train_rmses = []
val_rmses = []

for ln_lam in ln_lam_range:
    lam = np.exp(ln_lam)
    w = fit_regularized(Phi_train, t_train, lam)
    train_rmses.append(rmse(Phi_train @ w, t_train))
    val_rmses.append(rmse(Phi_val @ w, t_val))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ln_lam_range, train_rmses, color='#1a3a6e', linewidth=2, label='Training RMSE')
ax.plot(ln_lam_range, val_rmses, color='#cd0000', linewidth=2, label='Validation RMSE')
ax.set_xlabel(r'$\ln \lambda$')
ax.set_ylabel('RMSE')
ax.set_title('Regularization Path')
ax.legend()
fig.tight_layout()
plt.show()

## 6. Bias-Variance Decomposition

Expected loss = $(\text{bias})^2 + \text{variance} + \text{noise}$

We estimate this empirically by generating many datasets and averaging predictions.

In [ ]:
n_datasets = 200
N_bv = 25
M_bv = 9
x_eval = np.linspace(0.01, 0.99, 100)
t_eval_true = true_function(x_eval)

lambdas_bv = [0, np.exp(-6), np.exp(-3), np.exp(0)]
labels_bv = [r'$\lambda = 0$', r'$\lambda = e^{-6}$',
             r'$\lambda = e^{-3}$', r'$\lambda = e^{0}$']

results = {}

for lam, label in zip(lambdas_bv, labels_bv):
    predictions = np.zeros((n_datasets, len(x_eval)))

    for i in range(n_datasets):
        np.random.seed(i)
        x_d = np.sort(np.random.uniform(0, 1, N_bv))
        t_d = true_function(x_d) + np.random.normal(0, sigma, N_bv)

        Phi_d = gaussian_basis(x_d, M_bv, s=0.1)
        Phi_e = gaussian_basis(x_eval, M_bv, s=0.1)

        w = fit_regularized(Phi_d, t_d, lam)
        predictions[i] = Phi_e @ w

    avg_pred = predictions.mean(axis=0)
    bias_sq = np.mean((avg_pred - t_eval_true) ** 2)
    variance = np.mean(predictions.var(axis=0))

    results[label] = {'bias_sq': bias_sq, 'variance': variance,
                       'total': bias_sq + variance, 'predictions': predictions}
    print(f'{label:20s}  Bias^2 = {bias_sq:.4f}  Var = {variance:.4f}  '
          f'Total = {bias_sq + variance:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
colors_bv = ['#cd0000', '#FF8C00', '#228B22', '#1a3a6e']

for ax, (label, res), c in zip(axes.ravel(), results.items(), colors_bv):
    # Plot 20 sample fits
    for i in range(20):
        ax.plot(x_eval, res['predictions'][i], color=c, alpha=0.15, linewidth=0.8)

    avg = res['predictions'].mean(axis=0)
    ax.plot(x_eval, avg, color=c, linewidth=2.5, label='Average prediction')
    ax.plot(x_eval, t_eval_true, '--', color='black', linewidth=1.5,
            label=r'$\sin(2\pi x)$')
    ax.set_xlim(0, 1)
    ax.set_ylim(-1.8, 1.8)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'{label}  (Bias$^2$={res["bias_sq"]:.3f}, Var={res["variance"]:.3f})')
    ax.legend(fontsize=7)

fig.suptitle('Bishop Ch 4: Bias-Variance Decomposition', fontsize=13, y=1.02)
fig.tight_layout()
save_fig(fig, 'bishop_ch4_bias_variance')
plt.show()

## 7. Bias-Variance Trade-off Summary

In [ ]:
ln_lam_sweep = np.linspace(-30, 5, 60)
bias_curve = []
var_curve = []

for ln_lam in ln_lam_sweep:
    lam = np.exp(ln_lam)
    preds = np.zeros((n_datasets, len(x_eval)))

    for i in range(n_datasets):
        np.random.seed(i)
        x_d = np.sort(np.random.uniform(0, 1, N_bv))
        t_d = true_function(x_d) + np.random.normal(0, sigma, N_bv)
        Phi_d = gaussian_basis(x_d, M_bv, s=0.1)
        Phi_e = gaussian_basis(x_eval, M_bv, s=0.1)
        w = fit_regularized(Phi_d, t_d, lam)
        preds[i] = Phi_e @ w

    avg = preds.mean(axis=0)
    bias_curve.append(np.mean((avg - t_eval_true) ** 2))
    var_curve.append(np.mean(preds.var(axis=0)))

bias_curve = np.array(bias_curve)
var_curve = np.array(var_curve)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ln_lam_sweep, bias_curve, color='#1a3a6e', linewidth=2, label=r'$(\mathrm{bias})^2$')
ax.plot(ln_lam_sweep, var_curve, color='#cd0000', linewidth=2, label='variance')
ax.plot(ln_lam_sweep, bias_curve + var_curve, '--', color='#228B22',
        linewidth=2, label='bias$^2$ + variance')
ax.set_xlabel(r'$\ln \lambda$')
ax.set_ylabel('Error')
ax.set_title('Bias-Variance Trade-off')
ax.legend()
fig.tight_layout()
plt.show()

## 8. Summary

**Key takeaways from Bishop Chapter 4:**

- Linear regression is linear in the parameters, not necessarily in the inputs -- basis functions provide flexibility.
- MLE solution has a closed-form via the pseudo-inverse (normal equations).
- Regularization (ridge regression) shrinks weights toward zero, trading increased bias for reduced variance.
- The bias-variance decomposition reveals why optimal model complexity depends on both the data and the regularization strength.
- Different basis functions (polynomial, Gaussian RBF, sigmoidal) offer different inductive biases.